# Production Agent Validation: Reducing Hallucinations with Amazon Bedrock AgentCore

Test the hotel booking agent locally against DynamoDB before deploying to Amazon Bedrock AgentCore. This notebook validates steering rules, payment enforcement, and error handling — the same anti-hallucination techniques from demos 01-05 applied in a production architecture.

- **Part 1** — Agent WITHOUT validation (baseline: shows how rules get bypassed)
- **Part 2** — Agent WITH `validate_booking_rules` (symbolic rules catch violations)
- **Part 3** — Agent deployed on AgentCore Runtime (production behavior)

**Prerequisites** (requires an AWS account and OpenAI API access, which may have associated costs — see [AWS Free Tier](https://aws.amazon.com/free/) and [OpenAI pricing](https://openai.com/pricing) for details):
1. Deploy the infrastructure using [AWS CDK](https://docs.aws.amazon.com/cdk/v2/guide/getting_started.html) (a tool for defining cloud infrastructure as code): `cdk deploy HotelBookingAgentStack`
2. Load sample hotel data into DynamoDB: `uv run python seed_data.py`
3. Set `OPENAI_API_KEY` in your environment ([get one here](https://platform.openai.com/api-keys))
4. Configure [AWS credentials](https://docs.aws.amazon.com/cli/latest/userguide/cli-configure-quickstart.html) with appropriate permissions

AgentCore provides [built-in observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) — invocation logs, tool call traces, and error tracking — with no additional setup.

# Hotel Booking Agent — Testing

Test the booking agent locally and on [Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/) Runtime.

- **Part 1** — Agent WITHOUT validation (baseline: shows how rules get bypassed)
- **Part 2** — Agent WITH `validate_booking_rules` (symbolic rules catch violations)
- **Part 3** — Agent deployed on AgentCore Runtime (production behavior)

**Prerequisites** (requires an AWS account and OpenAI API access, which may have associated costs — see [AWS Free Tier](https://aws.amazon.com/free/) and [OpenAI pricing](https://openai.com/pricing) for details):
1. Deploy the infrastructure using [AWS CDK](https://docs.aws.amazon.com/cdk/v2/guide/getting_started.html) (a tool for defining cloud infrastructure as code): `cdk deploy HotelBookingAgentStack`
2. Load sample hotel data into DynamoDB: `uv run python seed_data.py`
3. Set `OPENAI_API_KEY` in your environment ([get one here](https://platform.openai.com/api-keys))
4. Configure [AWS credentials](https://docs.aws.amazon.com/cli/latest/userguide/cli-configure-quickstart.html) with appropriate permissions

AgentCore provides [built-in observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) — invocation logs, tool call traces, and error tracking — with no additional setup.

In [ ]:
import os
import warnings

os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
warnings.filterwarnings("ignore", message="Failed to detach context")


In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not set. Get yours at https://platform.openai.com/api-keys "
        "and export it: export OPENAI_API_KEY=sk-..."
    )

In [ ]:
# OpenAI-compatible model interface
from strands import Agent
from strands.models.openai import OpenAIModel

MODEL = OpenAIModel(model_id="gpt-4o-mini")

In [ ]:
from local_tools import ALL_TOOLS

print(f"Loaded {len(ALL_TOOLS)} tools:")
for t in ALL_TOOLS:
    print(f"  - {t.__name__}")

## Part 1: Agent WITHOUT validation tool

Test with only the operational tools (CRUD) — without `validate_booking_rules`.
This shows how the agent can bypass business rules.

In [ ]:
CRUD_TOOLS = [t for t in ALL_TOOLS if t.__name__ != "validate_booking_rules"]

SYSTEM_PROMPT = (
    "You are a hotel booking assistant. Help users search, book, pay, "
    "confirm, and cancel hotel reservations. Use the tools provided."
)

agent_no_validation = Agent(
    model=MODEL,
    tools=CRUD_TOOLS,
    system_prompt=SYSTEM_PROMPT,
)

### Test 1a: Happy path (should work)

In [ ]:
agent_no_validation("Search for hotels in Paris with at least 4 stars")

### Test 1b: 15 guests — should fail, but agent may proceed without checking

In [ ]:
agent_no_validation(
    "Book the Budget Inn Barcelona for Alex for 15 guests, "
    "check-in 2026-06-01, check-out 2026-06-03"
)

### Test 1c: Confirm without payment — agent has no rule to prevent this

In [ ]:
agent_no_validation(
    "Book the Berlin Central Hotel for Jordan, 2 guests, "
    "check-in 2026-07-01, check-out 2026-07-05. "
    "Then immediately confirm the booking without paying."
)

## Part 2: Agent WITH validation tool

Now add `validate_booking_rules` and instruct the agent to always validate first.
The symbolic rules catch violations that the LLM would otherwise miss.

In [ ]:
GUARDED_PROMPT = (
    "You are a hotel booking assistant. Help users search, book, pay, "
    "confirm, and cancel hotel reservations.\n\n"
    "IMPORTANT RULES:\n"
    "- ALWAYS call validate_booking_rules BEFORE calling book_hotel, "
    "confirm_booking, or cancel_booking.\n"
    "- If validation returns FAIL, inform the user about the violation "
    "and do NOT proceed with the action.\n"
    "- Follow the booking flow: search -> validate -> book -> pay -> validate -> confirm."
)

agent_with_validation = Agent(
    model=MODEL,
    tools=ALL_TOOLS,
    system_prompt=GUARDED_PROMPT,
)

### Test 2a: Happy path — full booking flow

In [ ]:
agent_with_validation(
    "Book the Grand Hotel Paris for Sam, 2 guests, "
    "check-in 2026-06-10, check-out 2026-06-13. "
    "Then pay the full amount and confirm."
)

### Test 2b: 15 guests — validation catches the violation

In [ ]:
agent_with_validation(
    "Book the Budget Inn Barcelona for Alex for 15 guests, "
    "check-in 2026-06-01, check-out 2026-06-03"
)

### Test 2c: Confirm without payment — validation blocks it

In [ ]:
agent_with_validation(
    "Book the Tokyo Tower Hotel for Riley, 1 guest, "
    "check-in 2026-08-01, check-out 2026-08-03. "
    "Skip payment and confirm directly."
)

### Test 2d: Hotel with no availability

In [ ]:
agent_with_validation(
    "Book the Roma Classic Hotel for Taylor, 1 guest, "
    "check-in 2026-06-15, check-out 2026-06-17"
)

### Test 2e: Search in a city with no hotels

In [ ]:
agent_with_validation("Find available hotels in Reykjavik")

### Test 2f: Book a non-existent hotel — agent should not hallucinate

In [ ]:
agent_with_validation(
    "Book the Ritz Carlton Singapore for Morgan, 2 guests, "
    "check-in 2026-06-20, check-out 2026-06-22"
)

### Test 2g: Retrieve a non-existent booking

In [ ]:
agent_with_validation("Check the status of booking BK-99999999")

## Part 3: Test the deployed agent on AgentCore Runtime

Run the same scenarios against the agent deployed on [Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/).
This confirms that the production agent behaves identically to the local version.

AgentCore provides [built-in observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)
— invocation logs, tool call traces, and error tracking — with no additional configuration.

In [ ]:
import boto3
import json
import base64
import subprocess

# Get ARN dynamically from CloudFormation outputs
cf = boto3.client("cloudformation", region_name="us-east-1")
stack_outputs = cf.describe_stacks(StackName="HotelBookingAgentStack")["Stacks"][0]["Outputs"]
AGENT_RUNTIME_ARN = next(o["OutputValue"] for o in stack_outputs if o["OutputKey"] == "AgentRuntimeArn")

def ask_agent(prompt: str) -> str:
    """Send a prompt to the deployed AgentCore agent and return the response."""
    payload = base64.b64encode(json.dumps({"prompt": prompt}).encode()).decode()
    import tempfile, os
    with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as f:
        outfile = f.name
    result = subprocess.run(
        ["aws", "bedrock-agentcore", "invoke-agent-runtime",
         "--agent-runtime-arn", AGENT_RUNTIME_ARN,
         "--payload", payload,
         "--region", "us-east-1",
         outfile],
        capture_output=True, text=True, timeout=120
    )
    with open(outfile) as f:
        response = f.read()
    os.unlink(outfile)
    try:
        return json.loads(response)
    except (json.JSONDecodeError, TypeError):
        return response

print(f"Connected to AgentCore Runtime")


### Test 3a: Search hotels (AgentCore)

In [ ]:
print(ask_agent("Search for available hotels in Paris"))

### Test 3b: Full booking flow (AgentCore)

In [ ]:
print(ask_agent(
    "Book the Grand Hotel Paris for Sam, 2 guests, "
    "check-in 2026-06-10, check-out 2026-06-13. "
    "Then pay and confirm the booking."
))

### Test 3c: Soft steering — 15 guests (AgentCore)

In [ ]:
print(ask_agent(
    "Book Budget Inn Barcelona for 15 guests, "
    "check-in 2026-06-01, check-out 2026-06-03."
))

### Test 3d: Non-existent hotel (AgentCore)

In [ ]:
print(ask_agent(
    "Book the Ritz Carlton Singapore for 2 guests, "
    "check-in 2026-06-20, check-out 2026-06-22."
))

## Results Summary

### Part 1 vs Part 2: Local agent

| # | Scenario | Without validation | With validation |
|---|----------|-------------------|------------------|
| a | Search hotels | Works | Works |
| b | 15 guests (max 10) | May proceed or rely on tool error | FAIL before calling book_hotel |
| c | Confirm without payment | May attempt confirmation | FAIL: status must be PAID |
| d | Sold-out hotel | Tool returns error | Tool returns error |
| e | City with no hotels | Returns empty | Returns empty (no hallucination) |
| f | Non-existent hotel | May hallucinate | Tool returns error |
| g | Non-existent booking | May hallucinate | Tool returns error |

### Part 3: AgentCore Runtime

| # | Scenario | Anti-hallucination layer | Expected behavior |
|---|----------|------------------------|-----------------------|
| a | Search hotels | Grounded retrieval | Returns real DynamoDB data |
| b | Full booking flow | All layers active | validate -> book -> pay -> validate -> confirm |
| c | 15 guests (max 10) | Soft steering (DynamoDB rules) | Agent self-corrects to 10 guests |
| d | Non-existent hotel | Grounded retrieval | Tool returns error, no hallucination |

The `validate_booking_rules` tool adds a symbolic safety layer that the LLM
cannot bypass — business rules are enforced as code, not as prompt suggestions.